# Analysis (ticket C8)
Reads every JSONL in `runs/`, scores each row with `src/score.py`, and rebuilds the two plots. Re-run when new files arrive. Plots use the 150 `test` photos only.

In [ ]:
import sys
sys.path.insert(0, '../src')
import analysis
df = analysis.tidy()
df.groupby(['model', 'split']).size()

In [ ]:
df.to_csv('../results/tidy.csv', index=False)
df.head()

## ASR per cell, default rendering, no defence (Wilson 95%)

In [ ]:
fig = analysis.plot_asr_by_cell(df)
fig.savefig('../results/asr_by_cell.png', dpi=150)

## ASR against font size per cell (no defence, high contrast, on the object)

In [ ]:
fig = analysis.plot_asr_by_font(df)
fig.savefig('../results/asr_by_font.png', dpi=150)

## Figure 2 (C9): ASR against blur and JPEG strength, default rendering, Wilson 95%

In [ ]:
fig = analysis.plot_asr_by_defence(df)
fig.savefig('../paper/figures/fig2_defence_strength.pdf')
fig.savefig('../paper/figures/fig2_defence_strength.png', dpi=150)

## Figure 3 (C9): EasyOCR recall (A9) against ASR, per cell and rendering setting, no defence, Wilson 95%

In [ ]:
import pandas as pd
fig = analysis.plot_ocr_vs_asr(df, pd.read_csv('../results/ocr_read.csv'))
fig.savefig('../paper/figures/fig3_ocr_vs_asr.pdf', bbox_inches='tight')

## Table 1 (C9): attack success, utility, image quality and time per defence
C-on is the context cell and I-off the original instruction cell, which the pilot gate kept in Table 1. I-on is the instruction cell H2 has used since the pilot gate, because I-off never worked. PSNR and SSIM use the A10 per-pair rows of the same test photos at the default rendering. `results/defence_timing.csv` comes from `src/time_defences.py` and names the machine it ran on.

In [ ]:
import pandas as pd
tv = analysis.textvqa()
tab = analysis.table1(df, tv, pd.read_csv('../results/image_quality_rows.csv'), pd.read_csv('../results/defence_timing.csv'))
with open('../paper/tables/table1_tradeoff.tex', 'w') as f:
    f.write(analysis.table1_latex(tab))
tab

## Table 2 (C9): attack success before and after the attacker search, 150 test photos, Wilson 95%
For each defence B9 attacked: the default rendering through the defence (B7), the setting B9 chose (`configs/attacker.yaml`) through the defence (B10), and the chosen setting with no defence (B10).

In [ ]:
import yaml
chosen = yaml.safe_load(open('../configs/attacker.yaml'))['defences']
tab2 = analysis.table2(df, analysis.attacker_tidy(), chosen)
with open('../paper/tables/table2_attacker.tex', 'w') as f:
    f.write(analysis.table2_latex(tab2))
tab2

## Pre-registered tests (C10)
H1a, H1b, H2, H3 and H4 per model, as in `docs/pre-registration.md` with the pilot-gate changes. The readings the pre-registration leaves open are listed at the top of `src/hypotheses.py`.

In [ ]:
import hypotheses
tests, h3 = hypotheses.run(df, tv, pd.read_csv('../results/ocr_read.csv'))
tests.to_csv('../results/hypothesis_tests.csv', index=False)
h3.to_csv('../results/h3_by_setting.csv', index=False)
summary = hypotheses.verdicts(tests)
summary.to_csv('../results/hypothesis_summary.csv', index=False)
summary